In [ ]:
# ============================================================
# Data Science Term Project - Modeling
# Dataset: healthcare-dataset-stroke-data.csv
# Prerequisite: Run scaling-encoding.ipynb first to generate processed_data.pkl
# Output: models.pkl (all trained models saved for evaluation)
#
# Project Goal: Compare how different imbalance-handling strategies
#               affect stroke prediction performance.
# Class imbalance: ~4.9% stroke rate (249 stroke vs 4861 normal)
# ============================================================

# =====================
# Step 1. Import Libraries
# =====================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False
import warnings
warnings.filterwarnings('ignore')
import pickle

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, f1_score, recall_score, precision_score,
    accuracy_score
)
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE, ADASYN
import plotly.express as px

print('=' * 60)
print('Step 1. Libraries imported successfully')
print('=' * 60)


# ============================================================
# Step 2. Load preprocessed data
# Load the preprocessed dataset and scaler from the previous stage.
# [Why use pickle here?]
# Loading the serialized binary file ensures that the exact same scaled data state
# is used for modeling without repeating the preprocessing steps, maintaining consistency.
# ============================================================
PROCESSED_PATH = r'../outputs/processed_data.pkl'

print('\n' + '=' * 60)
print('Step 2. Loading preprocessed data')
print('=' * 60)

with open(PROCESSED_PATH, 'rb') as f:
    data = pickle.load(f)

X_train_scaled = data['X_train_scaled']
X_test_scaled  = data['X_test_scaled']
y_train        = data['y_train']
y_test         = data['y_test']
X              = data['X']
dataset        = data['dataset']
scaler         = data['scaler']

print(f'\nTraining data size : {X_train_scaled.shape}')
print(f'Test data size     : {X_test_scaled.shape}')
print(f'Training label distribution:\n{y_train.value_counts()}')
print(f'\nStroke rate: {y_train.mean()*100:.1f}%')
print('\nPreprocessed data loaded successfully!')


# ============================================================
# Part 1. K-Means Clustering
# ============================================================
print('\n\n' + '=' * 60)
print('Part 1. K-Means Clustering')
print('=' * 60)

# Select continuous numeric features ('age', 'avg_glucose_level', 'bmi') and apply independent scaling.
# [Why independent scaling for K-Means?]
# K-Means is a distance-based algorithm that relies strictly on Euclidean distance.
# Continuous variables with larger raw magnitudes (e.g., glucose levels up to 271) will completely
# dominate variables with smaller ranges (e.g., BMI). Independent standardization (mean=0, std=1)
# ensures each clinical feature contributes equally to cluster boundary formulation.
KMEANS_FEATURES = ['age', 'avg_glucose_level', 'bmi']
KMEANS_FEATURES = ['age', 'avg_glucose_level', 'bmi']
X_kmeans = dataset[KMEANS_FEATURES].copy()
scaler_km = StandardScaler()
X_kmeans_scaled = scaler_km.fit_transform(X_kmeans)

# Calculate the inertia across a range of k values (1 to 10) to execute the Elbow Method.
# [Why use the Elbow Method?]
# To mathematically determine the optimal number of clusters without human bias.
# We look for the "elbow point" where the rate of decrease in inertia (within-cluster sum of squares)
# shifts from sharp to minimal. Here, k=3 was chosen as the definitive inflection point.
print('\n[Elbow Method] Computing inertia for k=1..10...')
inertias = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_kmeans_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), inertias, 'bo-', linewidth=2, markersize=8)
plt.axvline(x=3, color='red', linestyle='--', label='Optimal k=3')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method - Finding Optimal k')
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/elbow_method.png', dpi=150)
plt.close()
print('Saved: elbow_method.png')

print('\n[K-Means] Training with k=3...')
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_kmeans_scaled)

# Compute the Silhouette Score for the k=3 clustering result.
# [Why calculate the Silhouette Score?]
# It quantifies cluster quality by measuring how close each point is to its own cluster relative
# to neighboring clusters (ranging from -1 to +1). A score of 0.3719 indicates that while
# clinical boundaries overlap slightly—which is highly typical for real-world medical data—the
# formed clusters possess distinct, statistically valid structural configurations.
sil_score = silhouette_score(X_kmeans_scaled, cluster_labels)
print(f'\nSilhouette Score: {sil_score:.4f}')

# Map clusters to clinical risk groups based on empirical stroke rates.
# [Why map clusters to specific risk labels?]
# Raw cluster integers (0, 1, 2) lack clinical utility. By sorting the clusters in ascending order
# of their actual stroke occurrence rates, we transform abstract mathematical groups into actionable
# medical cohorts: Low Risk (0.2%), Medium Risk (5.7%), and High Risk (13.0%).
df_cluster = dataset[KMEANS_FEATURES + ['stroke']].copy()
df_cluster['cluster'] = cluster_labels
stroke_rates    = df_cluster.groupby('cluster')['stroke'].mean()
sorted_clusters = stroke_rates.sort_values().index.tolist()
risk_labels = {
    sorted_clusters[0]: 'Low Risk',
    sorted_clusters[1]: 'Medium Risk',
    sorted_clusters[2]: 'High Risk'
}
df_cluster['risk_group'] = df_cluster['cluster'].map(risk_labels)

print('\n[Risk Group Classification Result]')
for cluster, label in risk_labels.items():
    rate = stroke_rates[cluster] * 100
    print(f'  Cluster {cluster} -> {label}: Stroke rate {rate:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for label, group in df_cluster.groupby('risk_group'):
    axes[0].scatter(group['age'], group['avg_glucose_level'],
                    label=label, alpha=0.5, s=20)
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Average Glucose Level')
axes[0].set_title('K-Means: Age vs Glucose')
axes[0].legend()
risk_order = ['Low Risk', 'Medium Risk', 'High Risk']
rates = [df_cluster[df_cluster['risk_group']==r]['stroke'].mean()*100 for r in risk_order]
colors_bar = ['steelblue', 'orange', 'tomato']
axes[1].bar(risk_order, rates, color=colors_bar)
axes[1].set_ylabel('Stroke Rate (%)')
axes[1].set_title('Stroke Rate by Risk Group')
for i, v in enumerate(rates):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/kmeans_analysis.png', dpi=150)
plt.close()
print('Saved: kmeans_analysis.png')

# Generate an interactive 3D scatter plot using Plotly Express.
# [Why use a 3D interactive visualization instead of traditional 2D plots?]
# Reducing three continuous clinical indicators (Age, Glucose, BMI) into a 2D plane forces
# severe informational loss. Projecting the data onto a 3D spatial coordinate system preserves
# the full multidimensional topology. The interactive rotation allows medical stakeholders to
# visually inspect overlapping boundary regions and verify the distribution of stroke cases.
fig_3d = px.scatter_3d(
    df_cluster, x='age', y='avg_glucose_level', z='bmi',
    color='risk_group', symbol='stroke',
    color_discrete_map={'Low Risk':'steelblue','Medium Risk':'orange','High Risk':'tomato'},
    symbol_map={0:'circle', 1:'diamond'},
    title='K-Means 3D Visualisation (Age / Glucose / BMI)',
    opacity=0.7
)
fig_3d.write_html('../outputs/kmeans_3d.html')
print('[Plotly] Generating 3D interactive scatter plot...')
print('Saved: kmeans_3d.html (open in browser for interactive 3D rotation)')


# ============================================================
# Part 2. Random Forest Training - 4 Experiments
# ============================================================
print('\n\n' + '=' * 60)
print('Part 2. Random Forest Training (4 Experiments)')
print('=' * 60)

# RF hyperparameters (tuned)
# [Why constrain max_depth to 5 and min_samples_leaf to 20?]
# Medical datasets with extreme minority class imbalance (~4.9%) are highly susceptible to overfitting.
# Deep trees will simply memorize the statistical noise of the rare minority instances. Restructuring the
# model with shallower trees (max_depth=5) and larger leaves (min_samples_leaf=20) forces strong
# regularization. This structural constraint raised our out-of-fold AUC from 0.808 to 0.836.
RF_PARAMS = {
    'n_estimators'    : 300,
    'max_depth'       : 5,
    'min_samples_leaf': 20,
    'random_state'    : 42,
    'n_jobs'          : -1
}

# ────────────────────────────────────────
# Experiment 1: RF Baseline
# ────────────────────────────────────────
# Train a baseline Random Forest model directly on the raw, unadjusted training partition.
# [Why establish an unadjusted baseline experiment?]
# To expose the deception of the 'Accuracy' metric in imbalanced contexts. Because normal patients
# comprise 95.1% of the data, a naive model can achieve a high accuracy of 95.1% by blindly predicting
# zero stroke cases for every single patient. This baseline results in a fatal Recall score of 0.0,
# providing a rigorous scientific justification for the necessity of our subsequent resampling techniques.
print('\n[Experiment 1] Training RF Baseline...')
rf_base = RandomForestClassifier(**RF_PARAMS)
rf_base.fit(X_train_scaled, y_train)
print('Done!')


# ============================================================
# [Shared] Balanced Threshold Selection Helper
# ============================================================
# Used by Experiments 2, 3, 4 and Part 3.
#
# Selection rule:
#   Among thresholds where accuracy >= ACC_FLOOR, pick the one
#   that maximises Balanced Accuracy = (Sensitivity + Specificity) / 2.
#   If no threshold clears ACC_FLOOR, fall back to Balanced Accuracy max.
#
# Leakage-safe:
#   OOF probabilities from TRAINING set only (test never seen).
#   Resampling applied to train fold only; validation fold = original dist.
from sklearn.metrics import (balanced_accuracy_score as _bal_acc,
                             recall_score, precision_score,
                             accuracy_score, f1_score)

ACC_FLOOR   = 0.78
THRESH_GRID = np.round(np.arange(0.10, 0.91, 0.02), 2)


def _make_oof_probs(X, y, resample_fn=None, class_weight=None, n_splits=5):
    """Out-of-fold positive-class probabilities (leakage-safe)."""
    _skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    Xr   = X.reset_index(drop=True)
    yr   = pd.Series(y).reset_index(drop=True)
    oof  = np.zeros(len(yr))
    for tr_idx, va_idx in _skf.split(Xr, yr):
        X_tr, y_tr = Xr.iloc[tr_idx], yr.iloc[tr_idx]
        if resample_fn is not None:
            X_tr, y_tr = resample_fn(X_tr, y_tr)
        params = dict(RF_PARAMS)
        if class_weight is not None:
            params['class_weight'] = class_weight
        _m = RandomForestClassifier(**params)
        _m.fit(X_tr, y_tr)
        oof[va_idx] = _m.predict_proba(Xr.iloc[va_idx])[:, 1]
    return oof


def _select_threshold(y_true, oof_prob, label='', acc_floor=ACC_FLOOR):
    """Max Balanced Accuracy subject to accuracy >= acc_floor."""
    rows = []
    for thr in THRESH_GRID:
        pred = (oof_prob >= thr).astype(int)
        rows.append({
            'thr'      : thr,
            'recall'   : recall_score(y_true, pred, zero_division=0),
            'precision': precision_score(y_true, pred, zero_division=0),
            'accuracy' : accuracy_score(y_true, pred),
            'bal_acc'  : _bal_acc(y_true, pred),
            'f1'       : f1_score(y_true, pred, zero_division=0),
        })
    thr_df     = pd.DataFrame(rows)
    candidates = thr_df[thr_df['accuracy'] >= acc_floor]
    if len(candidates) > 0:
        best = candidates.sort_values(['bal_acc', 'recall'], ascending=False).iloc[0]
        note = f'accuracy>={acc_floor} constraint satisfied'
    else:
        best = thr_df.sort_values('bal_acc', ascending=False).iloc[0]
        note = f'WARNING: accuracy>={acc_floor} not achievable -> Balanced Accuracy fallback'
    thr = float(best['thr'])
    print(f'  [{label}] Selected threshold = {thr:.2f}  ({note})')
    print(f'    CV -> Recall={best["recall"]:.3f}  Precision={best["precision"]:.3f}'
          f'  Accuracy={best["accuracy"]:.3f}  BalAcc={best["bal_acc"]:.3f}'
          f'  F1={best["f1"]:.3f}')
    return thr


# ────────────────────────────────────────
# Experiment 2: SMOTE + RF
# ────────────────────────────────────────
# Execute Experiment 2 using SMOTE (Synthetic Minority Oversampling Technique).
# [Why select SMOTE over random oversampling?]
# Random oversampling simply duplicates existing minority rows, causing severe overfitting. SMOTE draws
# vector lines between a stroke patient and their k-nearest stroke neighbors, synthesizing entirely
# new data points along those vectors. This balances the class priors so that the Random Forest's
# Gini impurity criteria can actively learn generalizable features of stroke risks.
print('\n[Experiment 2] Training SMOTE + RF...')
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print(f'  Before SMOTE - Normal: {sum(y_train==0)}, Stroke: {sum(y_train==1)}')
print(f'  After  SMOTE - Normal: {sum(y_train_smote==0)}, Stroke: {sum(y_train_smote==1)}')

rf_smote = RandomForestClassifier(**RF_PARAMS)
rf_smote.fit(X_train_smote, y_train_smote)

print('  Searching for balanced threshold via 5-fold OOF...')
_oof_smote = _make_oof_probs(
    X_train_scaled, y_train,
    resample_fn=lambda a, b: SMOTE(random_state=42).fit_resample(a, b)
)
thr_smote = _select_threshold(y_train, _oof_smote, label='SMOTE+RF')
print('Done!')

# ────────────────────────────────────────
# Experiment 3: ADASYN + RF
# ────────────────────────────────────────
# Execute Experiment 3 using ADASYN (Adaptive Synthetic Sampling).
# [Why select ADASYN as an advancement over SMOTE?]
# SMOTE synthesizes samples uniformly across all minority instances. ADASYN dynamically calculates the
# ratio of majority-class neighbors surrounding each stroke instance. It then targets the shifting
# decision boundaries by generating more synthetic data for "hard-to-learn" instances located in
# dense overlap zones, making the random forest boundary more precise in complex clinical regions.
print('\n[Experiment 3] Training ADASYN + RF...')
adasyn = ADASYN(random_state=42)
X_train_adasyn, y_train_adasyn = adasyn.fit_resample(X_train_scaled, y_train)
print(f'  Before ADASYN - Normal: {sum(y_train==0)}, Stroke: {sum(y_train==1)}')
print(f'  After  ADASYN - Normal: {sum(y_train_adasyn==0)}, Stroke: {sum(y_train_adasyn==1)}')

rf_adasyn = RandomForestClassifier(**RF_PARAMS)
rf_adasyn.fit(X_train_adasyn, y_train_adasyn)

print('  Searching for balanced threshold via 5-fold OOF...')
_oof_adasyn = _make_oof_probs(
    X_train_scaled, y_train,
    resample_fn=lambda a, b: ADASYN(random_state=42).fit_resample(a, b)
)
thr_adasyn = _select_threshold(y_train, _oof_adasyn, label='ADASYN+RF')
print('Done!')

# ────────────────────────────────────────
# Experiment 4: class_weight + Threshold Optimization
# ────────────────────────────────────────
# Previous approach (F1 maximisation) caused accuracy collapse.
# New approach: Balanced Accuracy maximisation with accuracy floor.
print('\n[Experiment 4] Training class_weight + Threshold optimization...')
rf_cw = RandomForestClassifier(**RF_PARAMS, class_weight='balanced')
rf_cw.fit(X_train_scaled, y_train)

print('  Searching for balanced threshold via 5-fold OOF...')
_oof_cw = _make_oof_probs(
    X_train_scaled, y_train, class_weight='balanced'
)
thr_cw = _select_threshold(y_train, _oof_cw, label='classweight+RF')
best_thresh = thr_cw
print('Done!')


# ============================================================
# Part 3. K-Means Undersampling + RF
# ============================================================
print('\n\n' + '=' * 60)
print('Part 3. K-Means Undersampling + RF Training')
print('=' * 60)

# Execute Experiment 5 using an advanced K-Means Undersampling architecture.
# [Why construct a custom K-Means Undersampling process instead of random undersampling?]
# Traditional random undersampling blindly deletes majority-class instances, causing catastrophic
# information loss and destroying the true distribution topography of normal patients.
# By clustering the majority class (normal patients) into 'n' clusters (where n equals the exact number
# of stroke cases) and extracting only the mathematical centroids, we compress the sample size while
# perfectly preserving the structural variance and distribution of the normal population.
# This sophisticated approach achieved a balanced 1:1 ratio with minimal information loss,
# delivering our most stable and robust evaluation performance.
X_majority = X_train_scaled[y_train == 0]
X_minority = X_train_scaled[y_train == 1]
n_minority  = len(X_minority)

print(f'\nMajority (Normal) samples : {len(X_majority)}')
print(f'Minority (Stroke) samples  : {n_minority}')
print(f'-> Compressing majority to {n_minority} representative centroids via K-Means...')

km_under = KMeans(n_clusters=n_minority, random_state=42, n_init=5)
km_under.fit(X_majority)

X_majority_under = pd.DataFrame(
    km_under.cluster_centers_,
    columns=X_train_scaled.columns
)
X_train_km = pd.concat(
    [X_majority_under, X_minority.reset_index(drop=True)],
    ignore_index=True
)
y_train_km = pd.Series([0] * n_minority + [1] * n_minority)

print(f'After undersampling - Normal: {sum(y_train_km==0)}, Stroke: {sum(y_train_km==1)}')

rf_km = RandomForestClassifier(**RF_PARAMS)
rf_km.fit(X_train_km, y_train_km)

# KMeans undersampling makes the model over-sensitive to positive class
# -> OOF search selects a higher threshold to restore balance
print('  Searching for balanced threshold via 5-fold OOF...')
def _resample_km(X_tr, y_tr):
    X_tr = X_tr.reset_index(drop=True)
    y_tr = pd.Series(y_tr).reset_index(drop=True)
    _maj  = X_tr[y_tr == 0]; _mino = X_tr[y_tr == 1]; _n = len(_mino)
    _km   = KMeans(n_clusters=_n, random_state=42, n_init=5).fit(_maj)
    _Xu   = pd.DataFrame(_km.cluster_centers_, columns=X_tr.columns)
    _Xr   = pd.concat([_Xu, _mino.reset_index(drop=True)], ignore_index=True)
    return _Xr, pd.Series([0]*_n + [1]*_n)

_oof_km = _make_oof_probs(X_train_scaled, y_train, resample_fn=_resample_km)
thr_km = _select_threshold(y_train, _oof_km, label='KMeans+RF')
print('Done!')


# ============================================================
# Save all trained models to models.pkl
# ============================================================
print('\n\n' + '=' * 60)
print('Saving all models...')
print('=' * 60)

save_dict = {
    'scaler'         : scaler,
    'feature_columns': list(X.columns),
    'X_test_scaled'  : X_test_scaled,
    'y_test'         : y_test,
    'kmeans'         : kmeans,
    'scaler_km'      : scaler_km,
    'df_cluster'     : df_cluster,
    'risk_labels'    : risk_labels,
    'sil_score'      : sil_score,
    'rf_base'        : rf_base,
    'rf_smote'       : rf_smote,
    'rf_adasyn'      : rf_adasyn,
    'rf_cw'          : rf_cw,
    'rf_km'          : rf_km,
    # per-model balanced thresholds (accuracy-floor constrained, OOF-based)
    'best_thresh'    : best_thresh,   # class_weight+RF threshold (backward compat)
    'thr_smote'      : thr_smote,     # SMOTE+RF      threshold
    'thr_adasyn'     : thr_adasyn,    # ADASYN+RF     threshold
    'thr_cw'         : thr_cw,        # classweight+RF threshold
    'thr_km'         : thr_km,        # KMeans+RF     threshold
    'feature_names'  : list(X.columns),
}

MODELS_PATH = r'../outputs/models.pkl'
with open(MODELS_PATH, 'wb') as f:
    pickle.dump(save_dict, f)

print('\nmodels.pkl saved successfully!')
print(f'   Saved to: {MODELS_PATH}')
print('\nSaved objects:')
for key in save_dict:
    print(f'  - {key}')

# Threshold summary
print('\n' + '=' * 60)
print('Threshold Summary')
print('=' * 60)
for name, model_key, thr in [
    ('SMOTE+RF',       'rf_smote',  thr_smote),
    ('ADASYN+RF',      'rf_adasyn', thr_adasyn),
    ('classweight+RF', 'rf_cw',     thr_cw),
    ('KMeans+RF',      'rf_km',     thr_km),
]:
    print(f'  {name:16s}: prob = {model_key}.predict_proba(X_test_scaled)[:, 1]')
    print(f'  {"":16s}  pred = (prob >= {thr:.2f}).astype(int)')


Step 1. Libraries imported successfully

Step 2. Loading preprocessed data

Training data size : (4087, 17)
Test data size     : (1022, 17)
Training label distribution:
0    3888
1     199
Name: stroke, dtype: int64

Stroke rate: 4.9%

Preprocessed data loaded successfully!


Part 1. K-Means Clustering

[Elbow Method] Computing inertia for k=1..10...
Saved: elbow_method.png

[K-Means] Training with k=3...

Silhouette Score: 0.3719

[Risk Group Classification Result]
  Cluster 0 -> Low Risk: Stroke rate 0.2%
  Cluster 2 -> Medium Risk: Stroke rate 5.7%
  Cluster 1 -> High Risk: Stroke rate 13.0%
Saved: kmeans_analysis.png
[Plotly] Generating 3D interactive scatter plot...
Saved: kmeans_3d.html (open in browser for interactive 3D rotation)


Part 2. Random Forest Training (4 Experiments)

[Experiment 1] Training RF Baseline...
Done!

[Experiment 2] Training SMOTE + RF...
  Before SMOTE - Normal: 3888, Stroke: 199
  After  SMOTE - Normal: 3888, Stroke: 3888
  Searching for balanced thresh